# Stage 11 — DeepScoresV2 Dense Stage 9A Symbol-Region Preservation Evaluation

This is a **non-tuning, non-training** evaluation of the frozen `best.pt` checkpoint using DeepScoresV2 held-out ground-truth symbol annotations.
It samples annotation-centered symbol regions, applies a geometry-preserving camera-degradation assay, and compares degraded baseline versus restored output.
It creates no optimizer, performs no backpropagation, and verifies that model weights remain unchanged.

The result is a **symbol-region image-preservation proxy**, not OMR correctness or musical truth.
CUDA is preferred, CPU is allowed. Progress is atomically persisted to Drive so a Colab interruption can resume without keep-alive/idle-limit bypasses.


In [ ]:
from pathlib import Path
import hashlib, json, tarfile, random, math, io, os, time
from google.colab import drive
drive.mount('/content/drive')

A=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_DATA/ds2_dense.tar.gz")
O=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v1")
BEST=O/"best.pt"
X=Path("/content/st_score_restore_deepscoresv2_dense")
PROGRESS=O/"stage9a_symbol_region_progress.v1.json"
FINAL=O/"stage9a_symbol_region_final_evidence.json"
MD5="7237318e381e6e0848ec30eb82decb83"
SIZE=741814529
EXPECTED_BEST_SHA256="08b279161a9e8c4bd37376da221ecb4e07130724254ccf7d9591c8d32f368683"
EXPECTED_CONFIG="deff0f1270009839e234608dd9967038e1228a6b2b034e26059ac2f8cbfd0f80"
SEED=20260907
PER_CATEGORY=8
MAX_SAMPLES=1024
PATCH=128

if not A.exists():
    matches=list(Path("/content/drive/MyDrive").rglob("ds2_dense.tar.gz"))
    if len(matches)!=1:
        raise FileNotFoundError(f"Expected one ds2_dense.tar.gz, found {len(matches)}")
    A=matches[0]
if not BEST.exists():
    raise FileNotFoundError(BEST)
if A.stat().st_size!=SIZE:
    raise RuntimeError("Archive size mismatch")

def file_hash(path, algorithm):
    h=hashlib.new(algorithm)
    with path.open("rb") as f:
        while b:=f.read(8<<20):
            h.update(b)
    return h.hexdigest()

if file_hash(A,"md5")!=MD5:
    raise RuntimeError("Archive MD5 mismatch")
if file_hash(BEST,"sha256")!=EXPECTED_BEST_SHA256:
    raise RuntimeError("best.pt SHA256 mismatch")

def safe_extract():
    X.mkdir(parents=True,exist_ok=True)
    base=X.resolve()
    with tarfile.open(A,"r:gz") as t:
        for m in t.getmembers():
            p=m.name.replace("\\","/")
            if not p or p.startswith("/") or ".." in Path(p).parts or m.issym() or m.islnk() or m.isdev():
                raise RuntimeError(f"unsafe tar member: {m.name}")
            q=(X/p).resolve()
            if q!=base and base not in q.parents:
                raise RuntimeError(f"tar escape: {m.name}")
        t.extractall(X)

if not X.exists() or not any(X.iterdir()):
    safe_extract()

tests=list(X.rglob("deepscores_test.json"))
if len(tests)!=1:
    raise RuntimeError(f"Expected one deepscores_test.json, found {len(tests)}")
TEST_JSON=tests[0]
R=TEST_JSON.parent
payload=json.loads(TEST_JSON.read_text())

images=payload.get("images")
annotations=payload.get("annotations")
categories=payload.get("categories")
if not isinstance(images,list) or not isinstance(annotations,list):
    raise RuntimeError("Unsupported DeepScores test annotation structure")

def image_name(row):
    return row.get("file_name") or row.get("filename") or row.get("img_name") or row.get("name")

def image_id(row):
    return row.get("id") if row.get("id") is not None else row.get("img_id")

image_rows={}
for row in images:
    iid=image_id(row)
    name=image_name(row)
    if iid is not None and name:
        image_rows[str(iid)]=row
if len(image_rows)!=352:
    raise RuntimeError(f"Expected 352 official held-out image rows, got {len(image_rows)}")

def cat_name(cat_id):
    if isinstance(categories,dict):
        row=categories.get(str(cat_id), categories.get(cat_id))
        if isinstance(row,dict):
            return str(row.get("name") or row.get("label") or cat_id)
        if row is not None:
            return str(row)
    if isinstance(categories,list):
        for row in categories:
            rid=row.get("id") if isinstance(row,dict) else None
            if str(rid)==str(cat_id):
                return str(row.get("name") or row.get("label") or cat_id)
    return str(cat_id)

def ann_img_id(a):
    for k in ("image_id","img_id","imageId"):
        if k in a:
            return a[k]
    return None

def ann_cat_id(a):
    for k in ("category_id","cat_id","categoryId"):
        if k in a:
            return a[k]
    return None

def ann_bbox(a):
    b=a.get("bbox") or a.get("a_bbox") or a.get("aBBox")
    if not isinstance(b,(list,tuple)) or len(b)<4:
        return None
    return [float(x) for x in b[:4]]

def normalize_bbox(vals, img_w, img_h):
    x,y,a,b=vals
    if x>=0 and y>=0 and a>0 and b>0 and x+a<=img_w+2 and y+b<=img_h+2:
        return (x,y,a,b)
    if a>x and b>y and a<=img_w+2 and b<=img_h+2:
        return (x,y,a-x,b-y)
    return None

def image_path(name):
    for p in (R/"images"/name,R/name):
        if p.exists():
            return p
    matches=list(R.rglob(Path(name).name))
    if len(matches)==1:
        return matches[0]
    raise FileNotFoundError(name)

by_cat={}
from PIL import Image
for idx,a in enumerate(annotations):
    iid=ann_img_id(a); cid=ann_cat_id(a); raw=ann_bbox(a)
    if iid is None or cid is None or raw is None or str(iid) not in image_rows:
        continue
    row=image_rows[str(iid)]
    name=str(image_name(row))
    p=image_path(name)
    with Image.open(p) as im:
        bbox=normalize_bbox(raw,im.width,im.height)
    if bbox is None or bbox[2]<2 or bbox[3]<2:
        continue
    token=f"{cid}:{iid}:{idx}:{bbox}"
    key=hashlib.sha256(f"{SEED}:{token}".encode()).hexdigest()
    by_cat.setdefault(str(cid),[]).append((key,idx,str(iid),bbox))

selected=[]
for cid in sorted(by_cat):
    rows=sorted(by_cat[cid], key=lambda x:x[0])[:PER_CATEGORY]
    for _,idx,iid,bbox in rows:
        selected.append({"annotationIndex":idx,"imageId":iid,"categoryId":cid,"categoryName":cat_name(cid),"bbox":bbox})
selected=sorted(selected,key=lambda r:hashlib.sha256(f"{SEED}:{r['categoryId']}:{r['imageId']}:{r['annotationIndex']}".encode()).hexdigest())[:MAX_SAMPLES]
if not selected:
    raise RuntimeError("No valid symbol annotations selected")
selection_digest=hashlib.sha256(json.dumps(selected,sort_keys=True,separators=(",",":")).encode()).hexdigest()
print("Stage9A symbol-region samples:",len(selected),"categories:",len({r["categoryId"] for r in selected}),"selection:",selection_digest[:12])


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from PIL import Image,ImageFilter,ImageOps
from torchvision.transforms import functional as TF

D=torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_NAME=torch.cuda.get_device_name(0) if D.type=="cuda" else "CPU"

class C(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.n=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU(),nn.Conv2d(b,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU())
    def forward(self,x):
        return self.n(x)

class UNet(nn.Module):
    def __init__(self,b=32):
        super().__init__()
        self.e1=C(1,b); self.e2=C(b,2*b); self.e3=C(2*b,4*b); self.mid=C(4*b,8*b)
        self.d3=C(12*b,4*b); self.d2=C(6*b,2*b); self.d1=C(3*b,b); self.o=nn.Conv2d(b,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(F.max_pool2d(a,2)); c=self.e3(F.max_pool2d(b,2)); d=self.mid(F.max_pool2d(c,2))
        d=self.d3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode="bilinear",align_corners=False),c],1))
        d=self.d2(torch.cat([F.interpolate(d,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1))
        d=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1))
        return torch.clamp(x+torch.tanh(self.o(d))*.5,0,1)

m=UNet().to(D)
ck=torch.load(BEST,map_location=D)
if ck.get("md5")!=MD5 or ck.get("cfg")!=EXPECTED_CONFIG or ck.get("epoch")!=19:
    raise RuntimeError("Checkpoint identity mismatch")
m.load_state_dict(ck["model"]); m.eval()
sx=torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]],device=D).view(1,1,3,3); sy=sx.transpose(2,3)
def edge(x): return torch.sqrt(F.conv2d(x,sx,padding=1)**2+F.conv2d(x,sy,padding=1)**2+1e-6)
def state_sha256(model):
    h=hashlib.sha256()
    for name,t in sorted(model.state_dict().items()):
        h.update(name.encode()); h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
WEIGHTS_BEFORE=state_sha256(m)

def centered_symbol_patch(im,bbox):
    x,y,w,h=bbox; cx=x+w/2; cy=y+h/2; side=max(48.0,4.0*max(w,h))
    left=int(math.floor(cx-side/2)); top=int(math.floor(cy-side/2)); right=int(math.ceil(cx+side/2)); bottom=int(math.ceil(cy+side/2))
    pad_l=max(0,-left); pad_t=max(0,-top); pad_r=max(0,right-im.width); pad_b=max(0,bottom-im.height)
    if any((pad_l,pad_t,pad_r,pad_b)):
        im=ImageOps.expand(im,border=(pad_l,pad_t,pad_r,pad_b),fill=255); left+=pad_l; right+=pad_l; top+=pad_t; bottom+=pad_t
    return im.crop((left,top,right,bottom)).resize((PATCH,PATCH),Image.Resampling.BICUBIC)

def rng_for(row):
    token=f"{SEED}:stage9a:{row['categoryId']}:{row['imageId']}:{row['annotationIndex']}"
    return random.Random(int.from_bytes(hashlib.sha256(token.encode()).digest()[:8],"big"))

def degrade_symbol_patch(im,r):
    im=TF.adjust_brightness(im,r.uniform(.78,1.2)); im=TF.adjust_contrast(im,r.uniform(.82,1.18)); im=im.filter(ImageFilter.GaussianBlur(r.uniform(.15,1.75)))
    a=np.asarray(im).astype(np.float32)/255.; yy,xx=np.mgrid[:a.shape[0],:a.shape[1]]; cx,cy=r.uniform(0,a.shape[1]),r.uniform(0,a.shape[0])
    z=np.sqrt((xx-cx)**2+(yy-cy)**2); z/=max(z.max(),1); a*=1-r.uniform(0,.22)*(1-z); a+=np.random.default_rng(r.randrange(2**32)).normal(0,r.uniform(.002,.03),a.shape)
    a=np.clip(a,0,1); out=Image.fromarray((a*255).astype("uint8")); b=io.BytesIO(); out.save(b,"JPEG",quality=r.randint(52,95)); b.seek(0)
    return Image.open(b).convert("L")
def ten(im): return torch.from_numpy(np.asarray(im,dtype=np.float32)/255.).unsqueeze(0).unsqueeze(0)
def metrics(pred,target):
    pixel=F.l1_loss(pred,target).item(); edge_loss=F.l1_loss(edge(pred),edge(target)).item(); ink=(target<0.75); white=(target>0.95); pred_ink=(pred<0.85)
    ink_recall=((pred_ink & ink).float().sum()/ink.float().sum().clamp_min(1)).item(); false_ink=(((pred<0.80)&white).float().sum()/white.float().sum().clamp_min(1)).item()
    return {"pixelL1":pixel,"edgeLoss":edge_loss,"inkRecall":float(ink_recall),"falseInkRate":float(false_ink)}
def atomic_json(path,payload):
    tmp=path.with_suffix(path.suffix+".tmp"); tmp.write_text(json.dumps(payload,indent=2,sort_keys=True)); os.replace(tmp,path)

start=0; records=[]
if PROGRESS.exists():
    saved=json.loads(PROGRESS.read_text())
    if saved.get("artifactType")=="stage11_stage9a_symbol_region_progress" and saved.get("datasetMd5")==MD5 and saved.get("checkpointSha256")==EXPECTED_BEST_SHA256 and saved.get("configSha256")==EXPECTED_CONFIG and saved.get("selectionDigest")==selection_digest:
        records=list(saved.get("records") or []); start=len(records)
        if start>len(selected): raise RuntimeError("Progress record count exceeds selection")
        print("Resuming Stage9A symbol-region evaluation at",start,"/",len(selected))
    else: raise RuntimeError("Existing Stage9A progress identity mismatch")
print("Execution device:",DEVICE_NAME)


In [ ]:
with torch.no_grad():
    for i in range(start,len(selected)):
        row=selected[i]; imgrow=image_rows[row["imageId"]]; name=str(image_name(imgrow))
        with Image.open(image_path(name)) as raw: target_im=centered_symbol_patch(raw.convert("L"),row["bbox"])
        source_im=degrade_symbol_patch(target_im,rng_for(row)); x=ten(source_im).to(D); y=ten(target_im).to(D)
        if D.type=="cuda":
            with torch.autocast("cuda",dtype=torch.float16): p=m(x)
        else: p=m(x)
        records.append({"annotationIndex":row["annotationIndex"],"imageId":row["imageId"],"categoryId":row["categoryId"],"categoryName":row["categoryName"],"baseline":metrics(x,y),"restored":metrics(p,y)})
        if len(records)%16==0 or len(records)==len(selected):
            atomic_json(PROGRESS,{"artifactType":"stage11_stage9a_symbol_region_progress","datasetMd5":MD5,"checkpointSha256":EXPECTED_BEST_SHA256,"configSha256":EXPECTED_CONFIG,"selectionDigest":selection_digest,"records":records})
            print(f"progress {len(records)}/{len(selected)} ({100*len(records)/len(selected):.1f}%)")
WEIGHTS_AFTER=state_sha256(m); weights_mutated=WEIGHTS_AFTER!=WEIGHTS_BEFORE
def avg(items,key,side): return sum(float(r[side][key]) for r in items)/len(items)
global_baseline={k:avg(records,k,"baseline") for k in ("pixelL1","edgeLoss","inkRecall","falseInkRate")}
global_restored={k:avg(records,k,"restored") for k in ("pixelL1","edgeLoss","inkRecall","falseInkRate")}
by_category={}
for r in records: by_category.setdefault(r["categoryName"],[]).append(r)
category_summary={}; severe=[]
for name,rows in sorted(by_category.items()):
    b={k:avg(rows,k,"baseline") for k in ("pixelL1","edgeLoss","inkRecall","falseInkRate")}; q={k:avg(rows,k,"restored") for k in ("pixelL1","edgeLoss","inkRecall","falseInkRate")}
    bad=(len(rows)>=4 and q["edgeLoss"]>b["edgeLoss"]*1.05 and q["inkRecall"]<b["inkRecall"]-0.03)
    if bad: severe.append(name)
    category_summary[name]={"samples":len(rows),"baseline":b,"restored":q,"severeRegression":bad}
improved_fraction=sum(1 for r in records if r["restored"]["pixelL1"]<r["baseline"]["pixelL1"] and r["restored"]["edgeLoss"]<r["baseline"]["edgeLoss"])/len(records)
pass_policy={"globalPixelL1Improved":global_restored["pixelL1"]<global_baseline["pixelL1"],"globalEdgeLossImproved":global_restored["edgeLoss"]<global_baseline["edgeLoss"],"inkRecallNotMateriallyWorse":global_restored["inkRecall"]>=global_baseline["inkRecall"]-0.005,"falseInkRateNotMateriallyWorse":global_restored["falseInkRate"]<=global_baseline["falseInkRate"]+0.005,"sampleImprovedFractionAtLeast60Percent":improved_fraction>=0.60,"noSevereCategoryRegression":len(severe)==0,"weightsUnchanged":not weights_mutated}
stage9a_pass=all(pass_policy.values())
evidence={"artifactType":"stage11_deepscoresv2_dense_stage9a_symbol_region_evidence","datasetId":"deepscoresv2.dense.v2","archiveMd5":MD5,"checkpointSha256":EXPECTED_BEST_SHA256,"checkpointConfigSha256":EXPECTED_CONFIG,"checkpointEpoch":19,"executionDevice":DEVICE_NAME,"deviceType":D.type,"geometryPreservingAssay":True,"officialHeldOutImagesAvailable":352,"selectedSymbolRegions":len(records),"selectedCategories":len(by_category),"selectionDigest":selection_digest,"perCategoryLimit":PER_CATEGORY,"maxSamples":MAX_SAMPLES,"optimizerCreated":False,"backpropagationExecuted":False,"heldOutUsedForTraining":False,"heldOutUsedForTuning":False,"weightsBeforeSha256":WEIGHTS_BEFORE,"weightsAfterSha256":WEIGHTS_AFTER,"weightsMutated":weights_mutated,"globalBaseline":global_baseline,"globalRestored":global_restored,"sampleImprovedFraction":improved_fraction,"severeCategoryRegressions":severe,"categorySummary":category_summary,"passPolicy":pass_policy,"stage9aPreservationPass":stage9a_pass,"semanticScope":"ground_truth_annotation_centered_symbol_region_image_preservation_proxy","omrCorrectnessImplied":False,"musicalTruthImplied":False,"automaticFinalSelectionAuthorized":False,"finalStage11Pass":False,"productionInferenceAuthorized":False,"modelPublicationAuthorized":False,"stage12EntryAuthorized":False}
atomic_json(FINAL,evidence); print(json.dumps({k:v for k,v in evidence.items() if k!="categorySummary"},indent=2)); print("Saved:",FINAL)
